# 02 — Unified source acquisition and training-data integrity gate

This notebook streams the registered text, code, math, image, speech, video and temporal sources into the single OneCore corpus. It then validates three disjoint splits and ensures every modality feeds the same checkpoint and recurrent core.


In [ ]:
import os, sys, json
from pathlib import Path

from pathlib import Path

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs" / "phone-research.yaml").is_file() and (candidate / "src" / "onecore").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate OneCore project root from {current}")

root = find_project_root()
print("OneCore project root:", root)

sys.path.insert(0, str(root / "src"))
from onecore.acquire import acquire_sources
acquisition_report = acquire_sources(root, profile=os.environ.get("ONECORE_DATA_PROFILE", "initial"))
print("Acquisition summary:", json.dumps({
    "source_count": acquisition_report["source_count"],
    "new_records": acquisition_report["new_records"],
    "total_new_records": acquisition_report["total_new_records"],
    "sources": [{"source": item.get("source"), "status": item.get("status"), "read": item.get("read"), "accepted": item.get("accepted")} for item in acquisition_report["sources"]],
}, ensure_ascii=False, indent=2))
from onecore.data import validate_training_data, validate_validation_data, validate_test_data, validate_split_disjointness

train_report = validate_training_data(root)
validation_report = validate_validation_data(root)
test_report = validate_test_data(root)
disjointness_report = validate_split_disjointness(root)
report = {"train": train_report, "validation": validation_report, "test": test_report, "split_disjointness": disjointness_report}
print(json.dumps(report, ensure_ascii=False, indent=2))
print("Dataset gates: PASS")
